# Minimum Variance Allocator

**Goal**: long-only weights with the lowest possible portfolio volatility, given a covariance matrix. Will test the impact of denoising

Unlike the risk-weighted allocator from the baseline, which uses only each ETF's own
volatility, minimum variance uses the full covariance matrix: it looks for combinations
of ETFs whose correlations offset each other to reach the lowest portfolio volatility.

Because it inverts the covariance matrix, it is the allocator most sensitive to noise in
the estimated correlations.

It is run twice, identical except for the covariance:
- **min_var_raw:** sample covariance of the 504-day window
- **min_var_denoised:** covariance with denoised correlations (volatilities unchanged)

Any difference in turnover and risk-adjusted performance between the two is therefore
due to denoising. Weights are long-only and capped at 20%, as pre-registered.

In [25]:
# Imports
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from vportfolio.params import CAP, LOOKBACK
from vportfolio import load_data, engine, denoising

In [26]:
# build a function to run the allocator without denoising
# will return a series of weights indexed by tickers
def min_var(window, cap = CAP, denoise = False):
    if denoise:
    # use the array definition to keep the math fast
        cov = np.array(denoising.denoised_cov(window)) # take denoised cov if set to true
    else: 
        cov = np.array(window.cov()) # else take regular cov
    cov = cov * 252 * 10000 # annualised, in percent^2: same best weights, better-sized numbers, will allow the minimizer to run properly by scaling the covariance
    n = window.shape[1] # number of ETFs
    starting_weights = np.ones(n)/n # set the weights to equal values
    objective = lambda w: w @ cov @ w # sets the objective to the min var portfolio
    constraints = {"type": "eq", "fun": lambda w: w.sum() - 1} # the sum must add to one. the dictionary format is needed for scipy's minimize
    bounds = [(0, cap)] * n # set the cap between 0 and desired cap (20%)
    result = minimize(objective, starting_weights, method="SLSQP",
                  bounds=bounds, constraints=constraints,
                  options={"maxiter": 1000}) # use scipy minimizer with a safety margin
    if not result.success:
        raise ValueError(f"min_var optimisation failed: {result.message}") # Check that it worked before returning
    return pd.Series(result.x, index=window.columns)  

In [27]:
# tests for min_var

rng = np.random.default_rng(0)
window_two = pd.DataFrame(rng.normal(size=(10000, 2)) * [0.01, 0.02], columns=["A", "B"])
window_many = pd.DataFrame(rng.normal(size=(504, 10)) * np.linspace(0.002, 0.02, 10))

# 1. always true: valid weights, and no other valid portfolio has lower variance
w = min_var(window_many, cap=0.2)
assert np.isclose(w.sum(), 1)
assert (w >= -1e-8).all() and (w <= 0.2 + 1e-8).all()

cov_m = window_many.cov().values
best = w.values @ cov_m @ w.values
compared = 0
for _ in range(500):
    r = rng.dirichlet(np.ones(10) * 5) # random weights summing to 1, near even
    if (r <= 0.2).all(): # only portfolios that respect the cap
        assert best <= r @ cov_m @ r + 1e-12 # none beats the optimiser
        compared += 1
assert compared > 100 # enough portfolios were actually checked

# 2. known answer: two uncorrelated assets, vols 1% and 2%, no cap → 0.8 / 0.2
w = min_var(window_two, cap=1)
assert np.isclose(w["A"], 0.8, atol=0.02)
assert np.isclose(w["B"], 0.2, atol=0.02)

# 3. inputs matter: the cap binds, and denoising changes the weights
w = min_var(window_two, cap=0.6)
assert np.isclose(w["A"], 0.6, atol=0.01) # low-vol asset held at the cap
assert np.isclose(w["B"], 0.4, atol=0.01)
assert not np.allclose(min_var(window_many, denoise=True), min_var(window_many, denoise=False))

print("min_var passed all tests")

min_var passed all tests


Let's sanity check the min_var function works as intended on the 31-12-2021 window used before

In [28]:
# let's import the data
data = load_data.load_universe()
# let's take the last date of the research period
decision_date = data["rebal_dates"].iloc[-1]
# calculate the lookback window
window = engine.lookback_window(data["returns"], data["eligible"], decision_date, n=LOOKBACK)

w_raw = min_var(window, denoise=False)
w_dn = min_var(window, denoise=True)

comparison = pd.DataFrame({"raw": w_raw, "denoised": w_dn}).round(3)
comparison.sort_values("raw", ascending=False)

,raw,denoised
Ticker,,
SHY,0.200,0.200
UUP,0.200,0.200
IEF,0.198,0.193
FXF,0.181,0.170
FXY,0.104,0.131
HYG,0.047,0.051
DBC,0.031,0.015
FXA,0.021,0.000
TIP,0.009,0.014


On the 2021-12-31 window both versions concentrate in the lowest-volatility assets (SHY, UUP, IEF, FXF, FXY ≈ 88%), with SHY and UUP at the 20% cap. Denoising changes about 5% of the allocation, mostly in small positions. Its expected benefit, more stable weights over time, is tested in the backtest